# SolverNode 테스트 노트북

SolverNode의 TTA(Test Time Augmentation) 기반 추론을 테스트합니다.

## 테스트 항목
- 선지 셔플링 및 리맵핑
- Track A (비문학) / Track B (RAG) 분기
- 실제 데이터 일부에 대한 추론

In [1]:
import sys
import os
import random
import pandas as pd
import ast
import time
from collections import Counter

# 시드 설정
SEED = 42
random.seed(SEED)

# 프로젝트 루트 경로 추가
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.append(project_root)

from src.utils.config_loader import load_config

# config 로드
cfg = load_config()

In [2]:
# 실제 데이터 로드 시도
data_path = os.path.join(project_root, cfg.path.data.validate)

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f"✅ 데이터 로드 완료: {len(df)}개 샘플")
    print(f"🔹 컬럼: {list(df.columns)}")
    display(df.head(3))

✅ 데이터 로드 완료: 204개 샘플
🔹 컬럼: ['id', 'paragraph', 'problems', 'question_plus', 'parsed', 'question', 'choices', 'answer', 'len_total']


,id,paragraph,problems,question_plus,parsed,question,choices,answer,len_total
0,generation-for-nlp-673,I. 국제 문제에 대한 미국의 중요성 증가 II. 연방정부에 대한 대중 신뢰의 지속...,{'question': '1932년 이후 대통령직의 권력과 위신이 커졌다는 사실을 ...,NaN,{'question': '1932년 이후 대통령직의 권력과 위신이 커졌다는 사실을 ...,1932년 이후 대통령직의 권력과 위신이 커졌다는 사실을 설명하는 것은 다음 중 어...,"['I만', 'I 및 IV만', 'I, II 및 IV만', 'II 및 IV만']",3,207
1,generation-for-nlp-1360,“나는 그 후 바다 해안에 있는 예멘의 항구인 '아덴'으로 향했다. 산으로 둘러싸여...,{'question': '이븐 바투타의 경험은 다음 중 어떤 경험과 가장 안 비슷한...,NaN,{'question': '이븐 바투타의 경험은 다음 중 어떤 경험과 가장 안 비슷한...,이븐 바투타의 경험은 다음 중 어떤 경험과 가장 안 비슷한가?,"['정화', '마르코 폴로', '로렌초 데 메디치', '현장 삼장']",3,620
2,generation-for-nlp-1767,"서울시가 내년부터 전광판, 간판, 가로등 등 시내 옥외조명 130만여개를 직접 관리...",{'question': '서울시가 추진하는 옥외조명 관리 방안의 주된 목적은 무엇인...,NaN,{'question': '서울시가 추진하는 옥외조명 관리 방안의 주된 목적은 무엇인...,서울시가 추진하는 옥외조명 관리 방안의 주된 목적은 무엇인가?,"['빛 공해 방지 및 에너지 절감', '관광 활성화', '상업 광고 증가', '주거...",1,1606


In [3]:
# [Cell 3] SolverNode 로드
from src.agent.nodes.solver import SolverNode

print("⏳ SolverNode 초기화 중... (모델 로딩에 시간이 걸릴 수 있습니다)")
solver = SolverNode(cfg)
print("✅ SolverNode 초기화 완료!")

⏳ SolverNode 초기화 중... (모델 로딩에 시간이 걸릴 수 있습니다)
🔧 [SolverNode] 초기화 중... (사용 모델: main_solver)
🔄 [Loader] 모델 로딩 시작: Qwen3-32B-4bit (unsloth/Qwen3-32B-bnb-4bit)
   ↳ ⚡ 양자화 설정 적용 중...


/data/ephemeral/home/REPO_Subin/.venv/lib/python3.11/site-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ [Loader] 로딩 완료!
✅ SolverNode 초기화 완료!


In [4]:
# [Cell 4] 단일 문제 테스트

# 첫 번째 샘플 선택
sample = df.iloc[0]

# State 구성
state = {
    "paragraph": sample["paragraph"],
    "problem": {
        "question": sample["question"],
        "choices": sample["choices"] if isinstance(sample["choices"], list) else ast.literal_eval(sample["choices"])
    },
    "track_info": {
        "is_rag_required": False,
        "category": "비문학"
    }
}

print("📝 테스트 문제:")
print(f"   지문: {state['paragraph'][:100]}...")
print(f"   질문: {state['problem']['question']}")
print(f"   정답: {sample.get('answer', 'N/A')}")
print("\n" + "="*50)

# SolverNode 실행
start_time = time.time()
result = solver(state)
elapsed_time = time.time() - start_time

print("\n📊 결과:")
for res in result["solver_results"]:
    print(f" 풀이: {res['reasoning']}... → 원본답: {res['answer']}")

print(f"\n⏱️ 추론 시간: {elapsed_time:.2f}초 (문제당 {elapsed_time/len(result['solver_results']):.2f}초)")

📝 테스트 문제:
   지문: I. 국제 문제에 대한 미국의 중요성 증가 II. 연방정부에 대한 대중 신뢰의 지속적 향상 III. 연방정부의 책임을 확대시킨 뉴딜 등의 프로그램 IV. 대중매체의 발달로 인한 대...
   질문: 1932년 이후 대통령직의 권력과 위신이 커졌다는 사실을 설명하는 것은 다음 중 어느 것입니까?
   정답: 3

🤖 [Solver] TTA 기반 추론 시작 (Track: track_a, Choice N: 4)
   ▶️ [TTA V1] 추론 중... (순서: [3, 2, 4, 1])
{"reasoning": "뉴딜 프로그램은 1932년 이후의 사례이며, 지문 II는 권력 확대와 직접적인 관련이 없음. 따라서 I, III, IV가 관련되나 III은 선택지에 없으므로 I와 IV만 해당. 정답은 2번입니다.", "answer": 2}<|im_end|>
      ↳ 셔플된 답: 2 → 원본 답: 2
   ▶️ [TTA V2] 추론 중... (순서: [4, 3, 1, 2])
{"reasoning": "뉴딜 프로그램은 III에서 언급되어 대통령 권한 확대와 관련됨. I과 IV는 대통령의 가시성과 국제적 영향력 증가를 설명함. 따라서 정답은 I 및 IV만 해당하는 4번입니다.", "answer": 4}<|im_end|>
      ↳ 셔플된 답: 4 → 원본 답: 2
   ▶️ [TTA V3] 추론 중... (순서: [2, 4, 3, 1])
{"reasoning": "뉴딜 프로그램은 III에서 언급되어 대통령 권한 확대와 관련됨. I와 IV는 대통령의 국제적 역할과 가시성 증가를 설명함. II는 신뢰 향상은 권력 확대를 직접적으로 설명하지 않음.", "answer": 1}<|im_end|>
      ↳ 셔플된 답: 1 → 원본 답: 2
   ▶️ [TTA V4] 추론 중... (순서: [2, 3, 1, 4])
{"reasoning": "지문의 I, II, IV는 대통령 권한 강화와 관련된 요인을 설명하지만, II는 연방정부에 대한 신뢰를 언급

In [5]:
# [Cell 5] TTA 결과 분석

# 답안 분포 확인
answers = [res["answer"] for res in result["solver_results"]]
answer_counts = Counter(answers)

print("📊 TTA 답안 분포:")
for ans, count in sorted(answer_counts.items()):
    bar = "█" * count
    print(f"   {ans}번: {bar} ({count}표)")

# 다수결 결과
majority_answer = answer_counts.most_common(1)[0][0]
majority_votes = answer_counts.most_common(1)[0][1]
total_votes = len(answers)

print(f"\n🎯 다수결 결과: {majority_answer}번")

# TTA 일관성 분석
confidence = majority_votes / total_votes
unanimity = len(answer_counts) == 1  # 만장일치 여부

print(f"\n📈 TTA 일관성 분석:")
print(f"   투표 신뢰도: {confidence:.1%} ({majority_votes}/{total_votes})")
print(f"   만장일치: {'✅ 예' if unanimity else '❌ 아니오'}")
print(f"   답안 분산: {len(answer_counts)}개 선택지에 분포")

# 정답 비교 (정답이 있는 경우)
if "answer" in sample:
    correct = sample["answer"]
    is_correct = majority_answer == correct
    print(f"✅ 정답: {correct}번 → {'정답!' if is_correct else '오답'}") 

📊 TTA 답안 분포:
   2번: ████ (4표)
   3번: █ (1표)

🎯 다수결 결과: 2번

📈 TTA 일관성 분석:
   투표 신뢰도: 80.0% (4/5)
   만장일치: ❌ 아니오
   답안 분산: 2개 선택지에 분포
✅ 정답: 3번 → 오답


In [6]:
# [Cell 6] 여러 문제 배치 테스트
from collections import Counter

def run_solver_and_vote(solver, sample):
    """SolverNode 실행 후 다수결 투표"""
    state = {
        "paragraph": sample["paragraph"],
        "problem": {
            "question": sample["question"],
            "choices": sample["choices"] if isinstance(sample["choices"], list) else ast.literal_eval(sample["choices"])
        },
        "track_info": {
            "is_rag_required": "false",
            "category": "비문학"
        }
    }
    
    start = time.time()
    result = solver(state)
    elapsed = time.time() - start

    answers = [res["answer"] for res in result["solver_results"]]
    answer_counts = Counter(answers)
    majority = answer_counts.most_common(1)[0][0]
    confidence = answer_counts.most_common(1)[0][1] / len(answers)
    
    return majority, confidence, elapsed, result

# 테스트할 샘플 수 (데이터 크기에 맞게 조절)
num_samples = min(3, len(df))
correct_count = 0
total_time = 0
high_confidence_correct = 0
high_confidence_total = 0

print(f"🚀 {num_samples}개 문제 테스트 시작\n")

for idx in range(num_samples):
    sample = df.iloc[idx]
    predicted, confidence, elapsed, _ = run_solver_and_vote(solver, sample)
    total_time += elapsed
    
    conf_indicator = "🔥" if confidence >= 0.8 else "⚠️" if confidence >= 0.6 else "❓"
    
    if "answer" in sample:
        correct = sample["answer"]
        is_correct = predicted == correct
        if is_correct:
            correct_count += 1
        
        # 고신뢰도 문제 분석
        if confidence >= 0.8:
            high_confidence_total += 1
            if is_correct:
                high_confidence_correct += 1
        
        status = "✅" if is_correct else "❌"
        print(f"[{idx+1}/{num_samples}] {status} 예측: {predicted}번, 정답: {correct}번 | {conf_indicator} 신뢰도: {confidence:.0%} | ⏱️ {elapsed:.1f}초")
    else:
        print(f"[{idx+1}/{num_samples}] 예측: {predicted}번 | {conf_indicator} 신뢰도: {confidence:.0%} | ⏱️ {elapsed:.1f}초")

# 종합 결과
print("\n" + "="*60)
print("📊 종합 결과:")
if "answer" in df.columns:
    accuracy = correct_count / num_samples * 100
    print(f"   전체 정확도: {correct_count}/{num_samples} ({accuracy:.1f}%)")
    if high_confidence_total > 0:
        hc_accuracy = high_confidence_correct / high_confidence_total * 100
        print(f"   고신뢰도(≥80%) 정확도: {high_confidence_correct}/{high_confidence_total} ({hc_accuracy:.1f}%)")

print(f"   총 소요 시간: {total_time:.1f}초 (평균 {total_time/num_samples:.1f}초/문제)")

🚀 3개 문제 테스트 시작

🤖 [Solver] TTA 기반 추론 시작 (Track: track_b, Choice N: 4)
   ▶️ [TTA V1] 추론 중... (순서: [2, 4, 3, 1])
{"reasoning": "1932년 이후 대통령 권한이 확대된 것은 국제 문제(I)와 대중매체의 발달(IV)의 영향이 크다. 뉴딜은 연방정부의 역할을 확대했지만, 직접적으로 대통령의 권위를 높인 것은 아님. 대중 신뢰(II)는 권한 확대의 직접 원인이 아님.", "answer": 1}<|im_end|>
      ↳ 셔플된 답: 1 → 원본 답: 2
   ▶️ [TTA V2] 추론 중... (순서: [3, 2, 1, 4])
{"reasoning": "1932년 이후 대통령 권력의 증가는 국제 문제(I)와 대중매체의 발달(IV)에 의해 설명되며, 뉴딜은 연방정부의 역할을 확대했지만 대통령직의 권위 직접적으로 증가시키지는 않았다.", "answer": 2}<|im_end|>
      ↳ 셔플된 답: 2 → 원본 답: 2
   ▶️ [TTA V3] 추론 중... (순서: [2, 4, 1, 3])
{"reasoning": "1932년 이후 대통령 권력이 커진 이유는 국제 문제의 중요성(I)과 대중매체의 발달로 인한 가시성 확대(IV)가 주요 원인입니다. 연방정부의 신뢰가 향상되었다고 해서 대통령 권력이 직접적으로 커진 것은 아닙니다. 뉴딜은 연방정부의 역할을 확대했지만, 대통령 개인의 권위 증대를 직접적으로 설명하지는 않습니다.", "answer": 1}<|im_end|>
      ↳ 셔플된 답: 1 → 원본 답: 2
   ▶️ [TTA V4] 추론 중... (순서: [1, 3, 2, 4])
{"reasoning": "국제 문제의 중요성(I)과 대중매체의 발달(I)은 대통령의 역할과 가시성을 높였고, 이는 권력과 위신의 증대를 초래했다. 반면, 연방정부의 신뢰 향상(II)은 대통령 권한과 직접적인 관련이 적다. 뉴딜(III)은 연방정부의 역할 확대를 반영하지, 대통

In [7]:
# [Cell 7] 내부 함수 단위 테스트 (모델 로딩 없이)

print("🧪 SolverNode 내부 함수 테스트\n")

# 1. _format_choices 테스트
print("1️⃣ _format_choices 테스트:")
choices = ["선지A", "선지B", "선지C"]
formatted = solver._format_choices(choices)
print(f"   입력: {choices}")
print(f"   출력:\n{formatted}")

# 2. _parse_answer 테스트
print("\n2️⃣ _parse_answer 테스트:")
test_outputs = '{"reasoning": "분석 내용", "answer": 2}'
parsed_reasoning, parsed_answer = solver._parse_answer(test_outputs)
print(f"   '{test_outputs[:30]}...' → {parsed_answer}번")

# 3. _remap_index 테스트
print("\n3️⃣ _remap_index 테스트:")
shuffled_order = [3, 1, 4, 2, 5]  # 셔플된 1번 위치에 원본 3번이 있음
print(f"   셔플 순서: {shuffled_order}")
for i in range(1, 6):
    original = solver._remap_index(i, shuffled_order)
    print(f"   셔플 {i}번 → 원본 {original}번")

# 4. _generate_tta_versions 테스트
print("\n4️⃣ _generate_tta_versions 테스트:")
choices = ["A", "B", "C", "D", "E"]
versions = solver._generate_tta_versions(choices)
print(f"   원본 선지: {choices}")
print(f"   생성된 TTA 버전: {len(versions)}개")
for i, (shuffled, indices) in enumerate(versions):
    print(f"   V{i+1}: {shuffled} (순서: {indices})")

🧪 SolverNode 내부 함수 테스트

1️⃣ _format_choices 테스트:
   입력: ['선지A', '선지B', '선지C']
   출력:
1. 선지A
2. 선지B
3. 선지C

2️⃣ _parse_answer 테스트:
   '{"reasoning": "분석 내용", "answer...' → 2번

3️⃣ _remap_index 테스트:
   셔플 순서: [3, 1, 4, 2, 5]
   셔플 1번 → 원본 3번
   셔플 2번 → 원본 1번
   셔플 3번 → 원본 4번
   셔플 4번 → 원본 2번
   셔플 5번 → 원본 5번

4️⃣ _generate_tta_versions 테스트:
   원본 선지: ['A', 'B', 'C', 'D', 'E']
   생성된 TTA 버전: 5개
   V1: ['D', 'E', 'C', 'B', 'A'] (순서: [4, 5, 3, 2, 1])
   V2: ['A', 'C', 'E', 'D', 'B'] (순서: [1, 3, 5, 4, 2])
   V3: ['A', 'D', 'C', 'B', 'E'] (순서: [1, 4, 3, 2, 5])
   V4: ['C', 'D', 'E', 'B', 'A'] (순서: [3, 4, 5, 2, 1])
   V5: ['B', 'E', 'A', 'C', 'D'] (순서: [2, 5, 1, 3, 4])


In [8]:
# [Cell 6.5] Track B (RAG) 테스트

print("="*60)
print("🔍 Track B (RAG) 테스트")
print("="*60)

# Track B 테스트용 샘플 (문학/한국사 등 RAG가 필요한 카테고리 시뮬레이션)
sample = df.iloc[0]

# 가상의 RAG 컨텍스트 (실제 환경에서는 Retriever가 제공)
mock_retrieved_context = [
    "1932년 프랭클린 D. 루스벨트가 대통령에 당선되었다.",
    "뉴딜 정책은 대공황 극복을 위해 연방정부의 역할을 크게 확대시켰다.",
    "제2차 세계대전 이후 미국은 국제 문제에서 주도적 역할을 하게 되었다.",
    "대중매체의 발달로 대통령의 가시성과 영향력이 증가하였다."
]

# Track B State 구성
state_track_b = {
    "paragraph": sample["paragraph"],
    "problem": {
        "question": sample["question"],
        "choices": sample["choices"] if isinstance(sample["choices"], list) else ast.literal_eval(sample["choices"])
    },
    "track_info": {
        "is_rag_required": True,  # RAG 활성화
        "category": "한국사"
    },
    "retrieved_context": mock_retrieved_context  # RAG 결과
}

print("\n📝 Track B 테스트 문제:")
print(f"   카테고리: {state_track_b['track_info']['category']}")
print(f"   RAG 활성화: ✅")
print(f"   참고자료 수: {len(mock_retrieved_context)}개")
print(f"   질문: {state_track_b['problem']['question'][:50]}...")
print(f"   정답: {sample.get('answer', 'N/A')}")

print("\n📚 RAG 컨텍스트 미리보기:")
for i, ctx in enumerate(mock_retrieved_context[:2], 1):
    print(f"   [{i}] {ctx[:60]}...")

print("\n" + "-"*50)

# SolverNode 실행 (Track B)
start_time = time.time()
result_b = solver(state_track_b)
elapsed_time = time.time() - start_time

# 결과 분석
answers = [res["answer"] for res in result_b["solver_results"]]
answer_counts = Counter(answers)
majority_answer = answer_counts.most_common(1)[0][0]
confidence = answer_counts.most_common(1)[0][1] / len(answers)

print("\n📊 Track B 결과:")
print(f"   TTA 답안 분포: {dict(answer_counts)}")
print(f"   다수결 결과: {majority_answer}번 (신뢰도: {confidence:.0%})")
print(f"   추론 시간: {elapsed_time:.2f}초")

if "answer" in sample:
    correct = sample["answer"]
    is_correct = majority_answer == correct
    print(f"   정답 비교: {correct}번 → {'✅ 정답!' if is_correct else '❌ 오답'}")

🔍 Track B (RAG) 테스트

📝 Track B 테스트 문제:
   카테고리: 한국사
   RAG 활성화: ✅
   참고자료 수: 4개
   질문: 1932년 이후 대통령직의 권력과 위신이 커졌다는 사실을 설명하는 것은 다음 중 어느 것입...
   정답: 3

📚 RAG 컨텍스트 미리보기:
   [1] 1932년 프랭클린 D. 루스벨트가 대통령에 당선되었다....
   [2] 뉴딜 정책은 대공황 극복을 위해 연방정부의 역할을 크게 확대시켰다....

--------------------------------------------------
🤖 [Solver] TTA 기반 추론 시작 (Track: track_b, Choice N: 4)
   ▶️ [TTA V1] 추론 중... (순서: [4, 2, 1, 3])
{"reasoning": "1932년 루스벨트 대통령 당선과 뉴딜 정책은 연방정부의 역할을 확대(III), 대중매체 발달은 대통령의 가시성과 영향력을 키웠(IV). 또한, 제2차 세계대전 이후 미국의 국제적 주도력(I)도 대통령 권한 강화에 기여했다. 따라서 I, III, IV는 모두 관련되나, II는 직접 언급되지 않았다.", "answer": 4}<|im_end|>
      ↳ 셔플된 답: 4 → 원본 답: 3
   ▶️ [TTA V2] 추론 중... (순서: [1, 2, 3, 4])
{"reasoning": "1932년 루스벨트 대통령 당선 이후 뉴딜 정책과 대중매체의 발달이 대통령 권한과 가시성을 높였고, 제2차 세계대전 이후 미국의 국제적 주도력(I)도 대통령의 위신 증대에 기여했다. II는 지문에 언급되지 않았고, III은 권한 확대를 의미하지만 직접적으로 권위나 위신 증가를 설명하지는 않는다. 따라서 I 및 IV만 해당된다.", "answer": 2}<|im_end|>
      ↳ 셔플된 답: 2 → 원본 답: 2
   ▶️ [TTA V3] 추론 중... (순서: [3, 1, 4, 2])
{"reasoning": "1932년 루스벨

In [9]:
# [Cell 6.6] Track A vs Track B 비교 분석

print("="*60)
print("⚖️ Track A vs Track B 비교 분석")
print("="*60)

# 이전 셀들의 결과 사용 (result: Track A, result_b: Track B)
answers_a = [res["answer"] for res in result["solver_results"]]
answers_b = [res["answer"] for res in result_b["solver_results"]]

counts_a = Counter(answers_a)
counts_b = Counter(answers_b)

majority_a = counts_a.most_common(1)[0][0]
majority_b = counts_b.most_common(1)[0][0]

conf_a = counts_a.most_common(1)[0][1] / len(answers_a)
conf_b = counts_b.most_common(1)[0][1] / len(answers_b)

print("\n📊 비교 결과:")
print(f"┌{'─'*25}┬{'─'*15}┬{'─'*15}┐")
print(f"│ {'항목':<23} │ {'Track A':^13} │ {'Track B':^13} │")
print(f"├{'─'*25}┼{'─'*15}┼{'─'*15}┤")
print(f"│ {'다수결 답안':<21} │ {str(majority_a)+'번':^13} │ {str(majority_b)+'번':^13} │")
print(f"│ {'신뢰도':<22} │ {f'{conf_a:.0%}':^13} │ {f'{conf_b:.0%}':^13} │")
print(f"│ {'답안 분포':<21} │ {str(dict(counts_a)):^13} │ {str(dict(counts_b)):^13} │")
print(f"└{'─'*25}┴{'─'*15}┴{'─'*15}┘")

# 일치 여부 확인
if majority_a == majority_b:
    print(f"\n✅ Track A와 Track B의 다수결 결과가 일치합니다: {majority_a}번")
else:
    print(f"\n⚠️ Track A와 Track B의 결과가 다릅니다!")
    print(f"   Track A: {majority_a}번 vs Track B: {majority_b}번")
    print(f"   → RAG 컨텍스트가 추론에 영향을 미쳤을 수 있습니다.")

if "answer" in sample:
    correct = sample["answer"]
    print(f"\n🎯 정답: {correct}번")
    print(f"   Track A: {'✅ 정답' if majority_a == correct else '❌ 오답'}")
    print(f"   Track B: {'✅ 정답' if majority_b == correct else '❌ 오답'}")

⚖️ Track A vs Track B 비교 분석

📊 비교 결과:
┌─────────────────────────┬───────────────┬───────────────┐
│ 항목                      │    Track A    │    Track B    │
├─────────────────────────┼───────────────┼───────────────┤
│ 다수결 답안                │      2번       │      2번       │
│ 신뢰도                    │      80%      │      80%      │
│ 답안 분포                 │ {2: 4, 3: 1}  │ {3: 1, 2: 4}  │
└─────────────────────────┴───────────────┴───────────────┘

✅ Track A와 Track B의 다수결 결과가 일치합니다: 2번

🎯 정답: 3번
   Track A: ❌ 오답
   Track B: ❌ 오답
